In [1]:
# Prepare scraping tools and reuse saved page instead of downloading it again
# The scraper warned me about rows it couldn’t read. I checked, found they were 
# skiers who crashed in run 2, so they had 6-piece rows not 12 so I fixed it
import os, re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (student project: ski-race-analysis)"}
columns = ["rank", "bib", "fis_code", "name", "birth_year", "nation",
           "run1", "run2", "total", "diff", "race_points", "points_2"]
status_pattern = re.compile(r"did not|disqualified|not permitted", re.IGNORECASE)

def get_status(row):
    heading = row.find_previous(string=status_pattern)
    return heading.strip() if heading else "Finished"

def to_seconds(t):
    if pd.isna(t):
        return None
    if ":" in t:
        minutes, seconds = t.split(":")
        return int(minutes) * 60 + float(seconds)
    return float(t)

def get_html(raceid):
    path = f"../data/raw/race_{raceid}.html"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    url = f"https://www.fis-ski.com/DB/general/results.html?sectorcode=AL&raceid={raceid}"
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    with open(path, "w", encoding="utf-8") as f:
        f.write(response.text)
    time.sleep(2)
    return response.text

def scrape_race(raceid):
    soup = BeautifulSoup(get_html(raceid), "html.parser")
    rows = soup.select("a.table-row")
    athlete_rows = [r for r in rows if r.get("href") and "athlete" in r.get("href")]

    records = []
    for row in athlete_rows:
        p = list(row.stripped_strings)
        if len(p) == 12:
            record = dict(zip(columns, p))
        elif len(p) == 5:
            record = dict(zip(["bib", "fis_code", "name", "birth_year", "nation"], p))
        elif len(p) == 6:
            record = dict(zip(["bib", "fis_code", "name", "birth_year", "nation", "run1"], p))
        else:
            print(f"Race {raceid}: skipped a row with {len(p)} pieces: {p}")
            continue
        record["status"] = get_status(row)
        record["raceid"] = raceid
        records.append(record)

    df = pd.DataFrame(records)
    if df.empty or "total" not in df:
        return pd.DataFrame()
    for col in ["run1", "run2", "total"]:
        df[col] = df[col].apply(to_seconds)
    df["pct_behind"] = (df["total"] / df["total"].min() - 1) * 100
    return df

In [2]:
# Test on first race
test = scrape_race(131070)
print(len(test))

43


In [3]:
# Scrape every race, keep a list of races with no results and races that failed
# 428 races → 18,429 skiers from 351 races (77 races had no results)
race_ids = pd.read_csv("../data/raw/race_ids.csv")["raceid"].tolist()
all_results, empty, failed = [], [], []

for i, raceid in enumerate(race_ids):
    try:
        df = scrape_race(raceid)
        if df.empty:
            empty.append(raceid)
        else:
            all_results.append(df)
    except Exception as e:
        failed.append((raceid, str(e)))
    if (i + 1) % 25 == 0:
        print(f"{i+1}/{len(race_ids)} races done")

results = pd.concat(all_results, ignore_index=True)
results.to_csv("../data/raw/results.csv", index=False)
print(len(results), "rows |", len(all_results), "races |", len(empty), "empty |", len(failed), "failed")

Race 127292: skipped a row with 11 pieces: ['1', '12', '45547', 'GILSON Madison', '2008', 'AUS', '55.79', '55.73', '1:51.52', '1:51.52', '440.00']
Race 127292: skipped a row with 11 pieces: ['2', '11', '45543', 'GORMAN Tiffany', '2007', 'AUS', '56.57', '56.46', '1:53.03', '+1.51', '453.68']
25/428 races done
Race 128323: skipped a row with 7 pieces: ['7', '56792', 'MAIR Allegra', '2007', 'AUT', '58.36', '59.89']
50/428 races done
75/428 races done
100/428 races done
125/428 races done
150/428 races done
175/428 races done
200/428 races done
225/428 races done
250/428 races done
275/428 races done
300/428 races done
325/428 races done
350/428 races done
375/428 races done
400/428 races done
Race 128498: skipped a row with 10 pieces: ['1', '1', '715204', 'BALIC Selma', '2009', 'BIH', '1:02.55', '1:00.76', '2:03.31', '2:03.31']
Race 128498: skipped a row with 10 pieces: ['2', '2', '235459', 'TOTTA Nefeli', '2009', 'GRE', '1:05.77', '1:06.38', '2:12.15', '+8.84']
Race 131445: skipped a row

In [4]:
# For each empty race, count how many skiers its page has (0 = probably cancelled)
for raceid in empty[:20]:
    with open(f"../data/raw/race_{raceid}.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    skier_rows = [r for r in soup.select("a.table-row") if r.get("href") and "athlete" in r.get("href")]
    print(raceid, len(skier_rows))

127190 0
127202 0
127186 0
127188 0
127216 0
127220 0
127159 0
127161 0
127234 0
127236 0
128156 0
128158 0
128221 0
128222 0
130635 0
127970 0
127972 0
130325 0
130333 0
129477 0


In [ ]:
# Check all 77 empty races, how many have skiers on their page? (should be 0)
with_skiers = []
for raceid in empty:
    with open(f"../data/raw/race_{raceid}.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    skier_rows = [r for r in soup.select("a.table-row") if r.get("href") and "athlete" in r.get("href")]
    if len(skier_rows) > 0:
        with_skiers.append(raceid)
print(len(with_skiers), with_skiers)

2 [128498, 131445]


In [7]:
# Look at the 2 empty races that have skiers 
from collections import Counter
for raceid in with_skiers:
    with open(f"../data/raw/race_{raceid}.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    skier_rows = [r for r in soup.select("a.table-row") if r.get("href") and "athlete" in r.get("href")]
    print(raceid, Counter(len(list(r.stripped_strings)) for r in skier_rows))
    print(list(skier_rows[0].stripped_strings))

128498 Counter({10: 2})
['1', '1', '715204', 'BALIC Selma', '2009', 'BIH', '1:02.55', '1:00.76', '2:03.31', '2:03.31']
131445 Counter({10: 2})
['1', '2', '715204', 'BALIC Selma', '2009', 'BIH', '1:00.87', '1:04.12', '2:04.99', '2:04.99']


75 empty races are cancelled (0 skiers); 2 (128498, 131445) had only 2 skiers and no points → left out, too small to be useful